# Ordinal Değişken Dönüşümü

## 1. Ordinal Değişken Nedir?

Kategorik değişkenlerin iki türü vardı:
- **Nominal:** Kategoriler arasında sıralama yok (renk: kırmızı/mavi/yeşil)
- **Ordinal:** Kategoriler arasında doğal bir sıralama var, ama aralar 
  arasındaki "mesafe" eşit değil veya bilinmiyor (eğitim düzeyi: 
  ilkokul < lise < üniversite < yüksek lisans)

One-Hot Encoding nominal değişkenler için uygundu çünkü kategoriler 
arasında hiyerarşi yoktu. Ordinal bir değişkeni One-Hot ile kodlarsak 
sıralama bilgisini kaybederiz.

## 2. Neden Ayrı Bir Yöntem Gerekiyor?

Amaç: sıralamayı koruyan ama gereksiz yere "eşit aralık" varsayımı 
dayatmayan bir kodlama yapmak.

### a) Ordinal (Label) Encoding
Kategorilere sırayla tam sayı ata: ilkokul=1, lise=2, üniversite=3, 
yüksek lisans=4

- Basit ve yaygın
- Risk: Model sayıları aritmetik olarak yorumlayabilir (yüksek lisans - 
  lise = üniversite - ilkokul gibi bir eşitlik varsayımı)
- Ağaç tabanlı modellerde (Decision Tree, Random Forest, XGBoost) risk 
  daha düşük çünkü bu modeller eşik mantığıyla çalışır, mesafeyi değil 
  sırayı kullanır
- Doğrusal modellerde daha dikkatli olunmalı

### b) Domain Bilgisiyle Ağırlıklandırma
Bazen sıra sayıları yerine gerçek dünya bilgisine dayalı ağırlıklar 
kullanılır (ör. Likert ölçeklerinde Thurstone ölçekleme) — ileri seviye 
bir konu, şimdilik sadece değiniyoruz.

## 3. Pratikte Nasıl Yapılır (Python)

İki yaygın yöntem:
- `pandas.Categorical` ile `categories` sırası tanımlayıp `.cat.codes`
- `sklearn.preprocessing.OrdinalEncoder` ile `categories` parametresine 
  sırayı manuel vermek

## 4. Dikkat Edilmesi Gereken Tuzak

`OrdinalEncoder`'ı sıra belirtmeden kullanmak en sık yapılan hatadır. 
Sklearn varsayılan olarak **alfabetik sıraya** göre kodlar. "kötü", 
"iyi", "orta" gibi bir sütunda alfabetik sıra (iyi=0, kötü=1, orta=2) 
anlamsal sırayı tamamen bozar.

In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Senaryo: E-ticaret hizmet kalitesi anketi
# hizmet_kalitesi sütunu: "kötü", "orta", "iyi", "çok iyi"

# 1. Örnek DataFrame
veri = pd.DataFrame({
    'musteri_id': [101, 102, 103, 104, 105, 106, 107, 108],
    'hizmet_kalitesi': ['iyi', 'kötü', 'çok iyi', 'orta', 
                          'iyi', 'çok iyi', 'kötü', 'orta']
})

print("Orijinal veri:")
print(veri)
print()

# ------------------------------------------
# YÖNTEM 1: pandas.Categorical
# ------------------------------------------
sira = ['kötü', 'orta', 'iyi', 'çok iyi']

veri['hizmet_kalitesi_cat'] = pd.Categorical(
    veri['hizmet_kalitesi'],
    categories=sira,
    ordered=True
)

veri['hizmet_kalitesi_kod_pandas'] = veri['hizmet_kalitesi_cat'].cat.codes

# ------------------------------------------
# YÖNTEM 2: sklearn OrdinalEncoder
# ------------------------------------------
encoder = OrdinalEncoder(categories=[sira])
veri['hizmet_kalitesi_kod_sklearn'] = encoder.fit_transform(veri[['hizmet_kalitesi']])

print("Kodlanmış veri:")
print(veri)
print()

# ------------------------------------------
# EKSTRA: Sıra vermeden ne olur? (karşılaştırma için)
# ------------------------------------------
encoder_sirasiz = OrdinalEncoder()  # categories parametresi yok
veri['hizmet_kalitesi_kod_sirasiz'] = encoder_sirasiz.fit_transform(veri[['hizmet_kalitesi']])

print("Sıra verilmeden yapılan kodlama (yanlış örnek):")
print(veri[['hizmet_kalitesi', 'hizmet_kalitesi_kod_pandas', 'hizmet_kalitesi_kod_sklearn', 'hizmet_kalitesi_kod_sirasiz']])

Orijinal veri:
   musteri_id hizmet_kalitesi
0         101             iyi
1         102            kötü
2         103         çok iyi
3         104            orta
4         105             iyi
5         106         çok iyi
6         107            kötü
7         108            orta

Kodlanmış veri:
   musteri_id  ... hizmet_kalitesi_kod_sklearn
0         101  ...                         2.0
1         102  ...                         0.0
2         103  ...                         3.0
3         104  ...                         1.0
4         105  ...                         2.0
5         106  ...                         3.0
6         107  ...                         0.0
7         108  ...                         1.0

[8 rows x 5 columns]

Sıra verilmeden yapılan kodlama (yanlış örnek):
  hizmet_kalitesi  ...  hizmet_kalitesi_kod_sirasiz
0             iyi  ...                          0.0
1            kötü  ...                          1.0
2         çok iyi  ...                          

### Sonuç
Ordinal Encoder nesnesine sırayı vermezsek yukarıdaki gibi iyi = 0, kötü = 1, çok iyi = 3, orta = 2 gibi yanlış bir sonuç çıkar. Ordinal verilerin sayısal değişkenlere dönüştürülmesinde sırayı vermek bir zorunluluktur.